# Trabalho 2 — Apache Spark com MinIO e PostgreSQL
## Pipeline: PostgreSQL → landing-zone (CSV) → bronze (Delta Lake)

**Etapas:**
1. Configurar SparkSession com Delta Lake + S3/MinIO
2. Extrair tabelas do PostgreSQL e gravar na `landing-zone` em CSV
3. Ler os CSVs da `landing-zone` e gravar em Delta Lake no bucket `bronze`
4. Executar operações DML (INSERT, UPDATE, DELETE) nas tabelas Delta
5. Demonstrar Time Travel

## 1. Copia o driver PostgreSQL para dentro do PySpark
> ⚠️ **Rode esta célula apenas uma vez.** Depois de rodar, reinicie o kernel antes de continuar.

In [1]:
import subprocess
subprocess.run([
    r"C:\Users\jvol2\.local\bin\uv.exe",
    "pip", "install", "psycopg2-binary"
], check=True)

CompletedProcess(args=['C:\\Users\\jvol2\\.local\\bin\\uv.exe', 'pip', 'install', 'psycopg2-binary'], returncode=0)

In [2]:
import shutil
import pyspark
import os

src = r"C:\EngDados2\files\jars\postgresql-42.7.3.jar"
dst = os.path.join(pyspark.__path__[0], "jars", "postgresql-42.7.3.jar")

if not os.path.exists(src):
    raise FileNotFoundError(f"❌ JAR de origem não encontrado: {src}")

shutil.copy2(src, dst)
print(f"✅ JAR copiado para: {dst}")
print("\n🔄 REINICIE O KERNEL AGORA antes de continuar!")

✅ JAR copiado para: C:\EngDados2\files\.venv\Lib\site-packages\pyspark\jars\postgresql-42.7.3.jar

🔄 REINICIE O KERNEL AGORA antes de continuar!


## 2. Variáveis de ambiente e configurações
> Rode após reiniciar o kernel.

In [3]:
import os

os.environ["JAVA_HOME"]   = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"

MINIO_ENDPOINT   = "http://localhost:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin123"

PG_HOST     = "localhost"
PG_PORT     = "5432"
PG_DB       = "loja_db"
PG_USER     = "admin"
PG_PASSWORD = "admin123"
PG_URL      = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}"

LANDING_ZONE = "s3a://landing-zone"
BRONZE       = "s3a://bronze"
TABELAS      = ["produtos", "clientes", "vendas"]

print("✅ Configs OK")

✅ Configs OK


## 3. SparkSession com Delta Lake e MinIO (S3A)

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Trabalho2-SparkMinIO")
    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.12:3.2.0,"
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262"
    )
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint",               MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",             MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",             MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access",      "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    # ✅ Corrige o UnsatisfiedLinkError do NativeIO no Windows
    .config("spark.hadoop.fs.s3a.buffer.dir",             "C:/tmp/s3a")
    .config("spark.hadoop.fs.s3a.fast.upload",            "true")
    .config("spark.hadoop.fs.s3a.fast.upload.buffer",     "bytebuffer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("✅ Spark", spark.version)

✅ Spark 3.5.1


## 4. Extração: PostgreSQL → landing-zone (CSV)

In [5]:
import psycopg2

# Lê o init.sql
with open(r"C:\EngDados2\files\init.sql", "r", encoding="utf-8") as f:
    sql = f.read()

conn = psycopg2.connect(
    host="localhost", port=5432, dbname="loja_db",
    user="admin", password="admin123"
)
conn.autocommit = True
cur = conn.cursor()
cur.execute(sql)
print("✅ init.sql executado com sucesso!")

# Confirma as tabelas criadas
cur.execute("""
    SELECT table_schema, table_name 
    FROM information_schema.tables 
    WHERE table_type = 'BASE TABLE'
    AND table_schema NOT IN ('pg_catalog', 'information_schema')
    ORDER BY table_schema, table_name
""")
for row in cur.fetchall():
    print(f"  schema: {row[0]}  |  tabela: {row[1]}")

cur.close()
conn.close()

✅ init.sql executado com sucesso!
  schema: public  |  tabela: clientes
  schema: public  |  tabela: produtos
  schema: public  |  tabela: vendas


In [6]:
pg_props = {
    "user":     PG_USER,
    "password": PG_PASSWORD,
    "driver":   "org.postgresql.Driver"
}

for tabela in TABELAS:
    print(f"\n📥 Extraindo tabela: {tabela}")
    df = spark.read.jdbc(url=PG_URL, table=tabela, properties=pg_props)
    df.printSchema()
    df.show(5)
    destino = f"{LANDING_ZONE}/{tabela}"
    df.write.mode("overwrite").option("header", "true").csv(destino)
    print(f"✅ Tabela '{tabela}' gravada em: {destino}")

print("\n🎉 Extração concluída!")


📥 Extraindo tabela: produtos
root
 |-- id_produto: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- preco: decimal(10,2) (nullable = true)
 |-- estoque: integer (nullable = true)

+----------+----------------+-----------+-------+-------+
|id_produto|            nome|  categoria|  preco|estoque|
+----------+----------------+-----------+-------+-------+
|         1|   Notebook Dell|Informática|3500.00|     15|
|         2|  Mouse Logitech|Informática| 120.00|     80|
|         3|Teclado Mecânico|Informática| 800.00|     30|
|         4|  Monitor LG 24"|Informática|1500.00|     20|
|         5|       iPhone 15|    Celular|5500.00|      8|
+----------+----------------+-----------+-------+-------+
only showing top 5 rows

✅ Tabela 'produtos' gravada em: s3a://landing-zone/produtos

📥 Extraindo tabela: clientes
root
 |-- id_cliente: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- email: string (nullable = true)
 |

## 5. Transformação: landing-zone (CSV) → bronze (Delta Lake)

In [7]:
for tabela in TABELAS:
    print(f"\n🔄 Convertendo '{tabela}' para Delta Lake...")

    origem  = f"{LANDING_ZONE}/{tabela}"
    destino = f"{BRONZE}/{tabela}"

    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(origem)

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(destino)

    # Registra como tabela SQL
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tabela}_bronze
        USING DELTA
        LOCATION '{destino}'
    """)

    count = spark.read.format("delta").load(destino).count()
    print(f"✅ '{tabela}_bronze' criada com {count} registros em: {destino}")

print("\n🎉 Todas as tabelas estão no bucket bronze em formato Delta Lake!")


🔄 Convertendo 'produtos' para Delta Lake...
✅ 'produtos_bronze' criada com 64 registros em: s3a://bronze/produtos

🔄 Convertendo 'clientes' para Delta Lake...
✅ 'clientes_bronze' criada com 40 registros em: s3a://bronze/clientes

🔄 Convertendo 'vendas' para Delta Lake...
✅ 'vendas_bronze' criada com 80 registros em: s3a://bronze/vendas

🎉 Todas as tabelas estão no bucket bronze em formato Delta Lake!


## 6. DML — INSERT

In [8]:
print("=" * 50)
print("ESTADO ATUAL DA TABELA vendas_bronze")
print("=" * 50)
spark.sql("SELECT * FROM vendas_bronze ORDER BY id_venda").show()

# INSERT — novos registros de venda
spark.sql("""
INSERT INTO vendas_bronze VALUES
(100, 1, 3, 2, 120.00, '2024-03-01'),
(101, 2, 5, 1, 800.00, '2024-03-02')
""")

print("\n✅ INSERT executado!")
print("\nTabela APÓS o INSERT:")
spark.sql("SELECT * FROM vendas_bronze ORDER BY id_venda").show()

ESTADO ATUAL DA TABELA vendas_bronze
+--------+----------+----------+----------+--------------+----------+
|id_venda|id_cliente|id_produto|quantidade|valor_unitario|data_venda|
+--------+----------+----------+----------+--------------+----------+
|       1|         1|         1|         1|        3500.0|2024-01-15|
|       2|         2|         2|         3|         120.0|2024-01-16|
|       3|         3|         3|         1|         800.0|2024-01-17|
|       4|         1|         4|         2|        1500.0|2024-01-18|
|       5|         4|         5|         1|        5500.0|2024-01-19|
|       6|         5|         6|         4|         250.0|2024-01-20|
|       7|         2|         7|         2|         450.0|2024-02-01|
|       8|         3|         8|         1|         200.0|2024-02-02|
|       9|         4|         1|         1|        3500.0|2024-02-10|
|      10|         5|         2|         2|         120.0|2024-02-15|
|      11|         1|         1|         1|        35

## 7. DML — UPDATE

In [9]:
# UPDATE — corrige o valor_unitario da venda 100
spark.sql("""
UPDATE vendas_bronze
SET valor_unitario = 999.99
WHERE id_venda = 100
""")

print("✅ UPDATE executado! Novo valor da venda 100:")
spark.sql("SELECT * FROM vendas_bronze WHERE id_venda = 100").show()

✅ UPDATE executado! Novo valor da venda 100:
+--------+----------+----------+----------+--------------+----------+
|id_venda|id_cliente|id_produto|quantidade|valor_unitario|data_venda|
+--------+----------+----------+----------+--------------+----------+
|     100|         1|         3|         2|        999.99|2024-03-01|
+--------+----------+----------+----------+--------------+----------+



## 8. DML — DELETE

In [10]:
# DELETE — remove a venda 101
spark.sql("""
DELETE FROM vendas_bronze
WHERE id_venda = 101
""")

print("✅ DELETE executado!")
print("\nRegistros com id_venda >= 100 após DELETE:")
spark.sql("SELECT * FROM vendas_bronze WHERE id_venda >= 100").show()

✅ DELETE executado!

Registros com id_venda >= 100 após DELETE:
+--------+----------+----------+----------+--------------+----------+
|id_venda|id_cliente|id_produto|quantidade|valor_unitario|data_venda|
+--------+----------+----------+----------+--------------+----------+
|     100|         1|         3|         2|        999.99|2024-03-01|
+--------+----------+----------+----------+--------------+----------+



## 9. Time Travel — histórico de versões

In [11]:
vendas_path = f"{BRONZE}/vendas"

print("📜 Histórico de versões da tabela vendas:")
spark.sql("DESCRIBE HISTORY vendas_bronze") \
     .select("version", "timestamp", "operation") \
     .show(truncate=False)

📜 Histórico de versões da tabela vendas:
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|23     |2026-05-06 19:39:13|DELETE   |
|22     |2026-05-06 19:39:09|UPDATE   |
|21     |2026-05-06 19:39:06|WRITE    |
|20     |2026-05-06 19:38:59|WRITE    |
|19     |2026-05-06 19:35:51|DELETE   |
|18     |2026-05-06 19:35:48|UPDATE   |
|17     |2026-05-06 19:35:44|WRITE    |
|16     |2026-05-06 19:35:40|WRITE    |
|15     |2026-05-06 17:54:23|DELETE   |
|14     |2026-05-06 17:54:19|UPDATE   |
|13     |2026-05-06 17:54:15|WRITE    |
|12     |2026-05-06 17:54:10|WRITE    |
|11     |2026-05-05 19:03:58|DELETE   |
|10     |2026-05-05 19:03:51|UPDATE   |
|9      |2026-05-05 19:03:47|WRITE    |
|8      |2026-05-05 19:03:43|WRITE    |
|7      |2026-05-03 22:02:19|DELETE   |
|6      |2026-05-03 22:02:16|UPDATE   |
|5      |2026-05-03 22:02:13|WRITE    |
|4      |2026-05-03 22:02:09|WRITE    |
+-------+-------------------+---------+

In [12]:
# Leitura da versão 0 (estado original após ingestão)
print("🕐 Versão 0 — estado logo após a ingestão do CSV:")
spark.read \
    .format("delta") \
    .option("versionAsOf", 0) \
    .load(vendas_path) \
    .show()

🕐 Versão 0 — estado logo após a ingestão do CSV:
+--------+----------+----------+----------+--------------+----------+
|id_venda|id_cliente|id_produto|quantidade|valor_unitario|data_venda|
+--------+----------+----------+----------+--------------+----------+
|       1|         1|         1|         1|        3500.0|2024-01-15|
|       2|         2|         2|         3|         120.0|2024-01-16|
|       3|         3|         3|         1|         800.0|2024-01-17|
|       4|         1|         4|         2|        1500.0|2024-01-18|
|       5|         4|         5|         1|        5500.0|2024-01-19|
|       6|         5|         6|         4|         250.0|2024-01-20|
|       7|         2|         7|         2|         450.0|2024-02-01|
|       8|         3|         8|         1|         200.0|2024-02-02|
|       9|         4|         1|         1|        3500.0|2024-02-10|
|      10|         5|         2|         2|         120.0|2024-02-15|
|      11|         1|         1|         

## 10. Tabelas Gerenciadas vs Não Gerenciadas

| Aspecto | Gerenciada | Não Gerenciada (External) |
|---|---|---|
| Dados | Spark controla localização | Você define o path |
| DROP TABLE | Remove dados e metadados | Remove só metadados |
| Localização padrão | warehouse Spark | Qualquer path (ex: MinIO) |
| Uso típico | Tabelas temporárias | Data Lake / Produção |

Neste trabalho usamos tabelas **não gerenciadas** (external), pois os dados ficam no MinIO e registramos apenas os metadados no Spark com `LOCATION`.

In [13]:
# Verificando o tipo das tabelas criadas
spark.sql("DESCRIBE EXTENDED vendas_bronze") \
     .filter("col_name IN ('Type', 'Location', 'Provider')") \
     .show(truncate=False)

+--------+-------------------+-------+
|col_name|data_type          |comment|
+--------+-------------------+-------+
|Type    |EXTERNAL           |       |
|Location|s3a://bronze/vendas|       |
|Provider|delta              |       |
+--------+-------------------+-------+

